In [1]:
import json
import requests

# 1. "Static" Item Catalogue

In [2]:
# Parse items from static.json
STATIC_CATEGORIES = {
    "Currency", "Fragments", "Verisium", "Runes", "Expedition", "Vaal", "Delirium", "Breach", "Ritual", "Abyss", "Essences", "UncutGems", "LineageSupportGems", "Waystones"
}

class Item:
    """
    Item class to house individual item data
    """
    def __init__(self, id, text, category):
        self.id = id
        self.text = text
        self.category = category
    # Overwrite print function
    def __repr__(self):
        return f"Item(id={self.id!r}, text={self.text!r})"

def load_items_static(category):
    """
    Input:  category string
    Output: list of Item objects
    """
    with open("data/static.json") as f:
        data = json.load(f)

    items = []
    for cat in data["result"]:
        if cat["id"] == category:
            for i in cat["entries"]:
                # Exclude sep entries
                if i["id"] != "sep":
                    items.append(Item(id=i["id"], text=i["text"], category=category))
                # items.append(Item(id=i["id"], text=i["text"], category=category))

    if len(items) == 0:
        return None
    else:
        return items

# Load items into category:[items] dict
static_catalogue = {}
for cat in STATIC_CATEGORIES:
    static_catalogue[cat] = load_items_static(cat)

**Notes**
- The static.json file contains items which are eligible for trading in the official trade site's "Bulk Exchange" section.
- Each of these item types, excluding waystones, can also be traded using the in-game Currency Exchange
- The Currency Exchange provides the fastest turnaround. Minimal loading screens, trade window searching, and player-to-player interaction results in optimal profit

In [3]:
for cat in static_catalogue:
    print(f"{cat}:")
    for i in static_catalogue[cat]:
        print(f"\t{i.text} ({i.id})")

LineageSupportGems:
	Atalui's Bloodletting (ataluis-bloodletting)
	Einhar's Beastrite (einhars-beastrite)
	Brutus' Brain (brutus-brain)
	Uruk's Smelting (uruks-smelting)
	Paquate's Pact (paquates-pact)
	Vilenta's Propulsion (vilentas-propulsion)
	Ailith's Chimes (ailiths-chimes)
	Tecrod's Revenge (tecrods-revenge)
	Doedre's Undoing (doedres-undoing)
	Ratha's Assault (rathas-assault)
	Arjun's Medal (arjuns-medal)
	Kaom's Madness (kaoms-madness)
	Sione's Temper (siones-temper)
	Ixchel's Torment (ixchels-torment)
	Romira's Requital (romiras-requital)
	Tawhoa's Tending (tawhoas-tending)
	Dominus' Grasp (dominus-grasp)
	Dialla's Desire (diallas-desire)
	Kalisa's Crescendo (kalisas-crescendo)
	Kulemak's Dominion (kulemaks-dominion)
	Amanamu's Tithe (amanamus-tithe)
	Xoph's Pyre (xophs-pyre)
	Esh's Radiance (eshs-radiance)
	Tul's Stillness (tuls-stillness)
	Uul-Netol's Embrace (uul-netols-embrace)
	Rakiata's Flow (rakiatas-flow)
	Garukhan's Resolve (garukhans-resolve)
	Tasalio's Rhythm (tasal

# 2. Connect to External Source

**Potential Data Sources**
- https://www.pathofexile.com/trade2/search/poe2/Runes%20of%20Aldur
- https://www.pathofexile.com/trade2/exchange/poe2/Runes%20of%20Aldur
- https://mirrormarket.org/chart/perfect-exalted-orb/exalted
- trade data from official trade site does not reflect Currency Exchange prices. As per the official API documentation, they do provide a history of currency exchange prices, but this requires API access... which I do not have
- MirrorMarket can be used as an alternative. They track historical price changes for every currency exchange item, unlike poe.ninja which just uses the big 3 (exalt, chaos, divine)

In [4]:
# Session to reuse network connection (prevent multiple TLS handshakes)
session = requests.Session()
session.headers.update({"User-Agent": "personal-arbitrage-tool", "Accept": "application/json"})

def price_history(have, want, interval="1H", league="Runes of Aldur"):
    url = f"https://mirrormarket.org/api/price-history/{have}/{want}/{interval}"
    resp = session.get(url, params={"league": league}, timeout=10)
    # ignore 2xx (good) status codes, raise HTTPError on 4xx/5xx (bad) codes
    resp.raise_for_status()
    return resp.json()

def current_rate(have, want, interval="1H", league="Runes of Aldur"):
    records = price_history(have, want, interval, league)
    if not records:
        print("no records")
        return None
    return records[-1]["close"]

def get_price(have, want, interval="1H", league="Runes of Aldur"):
    rate = current_rate(have, want, interval, league)
    if not rate:
        "no records"
        return None
    if rate < 1:
        price = f"{round((1/rate), 2)}:1"
    else:
        price = f"1:{round(rate, 2)}"
    return price

*Response format*
- Responses use OHLC candlestick data
- time: when snapshot was taken (unix format)
- (O) open: rate at the start of the candlestick interval (first trade)
- (HL) high/low: extremes during the candlestick interval
- (C) close: rate at the end of the candlestick interval (last trade)
- volume: how many of the items were traded in the interval
- Volume in particular can be used to filter out outliers. If an item sells for a potentially profitable amount, but only one trade occurred in the interval period, it is not worth the time to seek that trade rate

In [9]:
h = "divine"
w = "exalted"

print(f"{h} -> {w} = {get_price(h, w)}")

divine -> exalted = 1:424.21
